# Bernini-R 指令式视频/图像编辑 — Colab G4 启动脚本

**模型**：字节跳动 Bernini-R（基于 Wan2.2 的统一编辑模型）　**运行时**：G4 高 RAM（RTX PRO 6000 Blackwell / 96GB）

全程用 ComfyUI **原生节点**，**不需要任何自定义节点**。

| 项 | 值 |
| --- | --- |
| 模型数量 | **7 个**（高噪专家 / 低噪专家 / 文本编码器 / VAE / 三个 LoRA），共约 **38GB** |
| 内网穿透 | FRP tcp 无 token，端口 **8092** |
| 访问地址 | http://usoren.usdream.dpdns.org:8092 |

按顺序跑 **Cell 0 → 5**。Cell 0 会升级 torch 并自动重启运行时，重启后从 Cell 1 继续。重启界面只需重跑 Cell 4 与 Cell 5。

> **Cell 0 必须先跑**：Colab 默认 torch 是 cu128，在 sm_120（Blackwell）上会让 comfy-kitchen 的 cuda / triton 融合后端全部 disabled，启动日志出现 `You need pytorch with cu130 or higher to use optimized CUDA operations`，走无优化回退路径，速度慢 1.5~3 倍且 fp8 更容易 OOM。

> **Bernini-R 是 Wan2.2 MoE 双专家架构**：必须同时下 `high_noise` 和 `low_noise` 两个主模型。官方文档里写的单个 `wan2.2_bernini_r_fp16.safetensors` 已过期，以工作流 JSON 为准。

> **关于 LoRA**：原版模板复用了一个 **Wan 2.1 T2V** 的 lightx2v LoRA，在高噪 3.0 / 低噪 1.5 上挂载。这是跨版本错配加超强度，会把模型拽回不含编辑条件的蒸馏先验，在 rv2v 上表现为「输出≈输入」。本笔记本改下 `rzgar/Bernini-R-LightX2V-4step-loras` 的 Bernini-R 专用高/低噪双 LoRA，强度均 1.0，配 4 步 / dpmpp_2m_sde / sgm_uniform。

> **服务端前提**：frps 配置里不能有 `auth.token`，改完 `systemctl restart frps`；8092 需空闲且在 `allowPorts` 内。

> **防断连**：在 Colab 页面按 F12 → Console 粘贴：`setInterval(()=>{document.body.click()},60000)`


In [ ]:
# ==========================================
# Cell 0: 升级 torch 到 cu130 (Blackwell sm_120 必需, 首次必跑)
# ------------------------------------------
# Colab 默认镜像是 torch 2.x + cu128。在 RTX PRO 6000 (sm_120) 上,
# comfy-kitchen 的 cuda / triton 融合后端会因为 cu128 全部 disabled,
# 启动日志里会看到:
#   WARNING: You need pytorch with cu130 or higher to use optimized CUDA operations
# 结果是走无优化回退路径, 速度慢 1.5~3 倍, 且 fp8 量化更容易 OOM。
#
# 本格自动判断, 只在需要时安装, 装完自动重启运行时。
# 重启后从 Cell 1 继续跑即可 (磁盘不丢, 已下载的模型还在)。
# ==========================================
import os
import time

FLAG = "/content/.cu130_installed"
IDX = "https://download.pytorch.org/whl/nightly/cu130"


def cuda_major():
    try:
        import torch
        if not torch.version.cuda:
            return 0
        return int(torch.version.cuda.split(".")[0])
    except Exception:
        return 0


major = cuda_major()
print("当前 torch CUDA 主版本: " + str(major))

if major >= 13:
    import torch
    print("OK 已是 cu130+ (" + torch.__version__ + "), 直接跑 Cell 1")
elif os.path.exists(FLAG):
    print("已尝试安装过但仍不是 cu130, 不再自动重启。请手动执行:")
    print("   !pip install --pre --force-reinstall torch torchvision torchaudio --index-url " + IDX)
else:
    print("安装 cu130 nightly 版 torch (约 3~5 分钟, 下载量大)...")
    os.system("pip install -q --pre torch torchvision torchaudio --index-url " + IDX)
    open(FLAG, "w").write("done")
    print("安装完成, 3 秒后自动重启运行时")
    print("重启是必须的, 否则进程里还是旧 torch。重启后请从 Cell 1 开始。")
    time.sleep(3)
    os.kill(os.getpid(), 9)


In [ ]:
# ==========================================
# Cell 1: 安装 / 强制更新 ComfyUI 到最新 nightly
# ------------------------------------------
# Bernini-R 的原生节点、以及模板库里的 Bernini-R 条目, 分属三个来源:
#   1. 节点代码  -> ComfyUI git 仓库 (comfy_extras/)
#   2. 前端界面  -> pip 包 comfyui-frontend-package
#   3. 模板库    -> pip 包 comfyui-workflow-templates
# 只做 git pull 不会更新后两者, 这就是模板库搜不到的原因。
# ==========================================
import os
import subprocess

REPO = "/content/ComfyUI"
print("=== 安装 / 更新 ComfyUI ===")
%cd /content

if not os.path.exists(REPO):
    !git clone https://github.com/comfyanonymous/ComfyUI

# git pull 遇到本地改动会失败且不报错, 改用 fetch + reset --hard 强制对齐远端
!cd {REPO} && git fetch --all --quiet && git reset --hard origin/master --quiet
!cd {REPO} && git log -1 --format="当前 ComfyUI 版本: %h  %ad  %s" --date=short

%cd /content/ComfyUI
# hf_transfer 已弃用 (环境变量 HF_HUB_ENABLE_HF_TRANSFER 也同时弃用), 改用 hf_xet
!pip install -q -r requirements.txt huggingface_hub hf_xet

# 前端包 / 模板库 / 内置文档 都是独立 pip 包, 必须单独升级
print("\n=== 升级前端包与模板库 ===")
!pip install -q -U comfyui-frontend-package comfyui-workflow-templates comfyui-embedded-docs

# ComfyUI-Manager 只作排错用, 本工作流不需要任何自定义节点
MGR = "/content/ComfyUI/custom_nodes/ComfyUI-Manager"
if not os.path.exists(MGR):
    !git clone -q https://github.com/ltdrdata/ComfyUI-Manager.git {MGR}

# ---------------- 自检 ----------------
print("\n=== 环境自检 ===")

import torch
print("torch:", torch.__version__, "| cuda:", torch.version.cuda)
if torch.cuda.is_available():
    cap = torch.cuda.get_device_capability(0)
    print("GPU:", torch.cuda.get_device_name(0), "| capability:", cap)
    if cap[0] >= 12:
        print("OK Blackwell, fp8 / fp16 均可直接跑")
else:
    print("未检测到 GPU, 请检查运行时类型")

# cu130 关卡: 决定 comfy-kitchen 能否启用融合内核
if torch.version.cuda and int(torch.version.cuda.split(".")[0]) >= 13:
    print("OK cu130+, comfy-kitchen 的 cuda / triton 后端可启用")
else:
    print("!! 当前不是 cu130。请先跑 Cell 0 升级 torch,")
    print("   否则融合内核全部 disabled, 速度慢 1.5~3 倍且更容易 OOM。")

import importlib.metadata as md
for pkg in ["comfyui-frontend-package", "comfyui-workflow-templates"]:
    try:
        print(pkg + ": " + md.version(pkg))
    except Exception:
        print(pkg + ": 未安装")

node_hit = subprocess.run(
    "grep -ril bernini /content/ComfyUI/comfy_extras/ | head -5",
    shell=True, capture_output=True, text=True).stdout.strip()
if node_hit:
    print("\nOK 找到 Bernini 原生节点:\n" + node_hit)
else:
    print("\n!! 未找到 Bernini 节点。处理: !rm -rf /content/ComfyUI 后重跑本格")

print("\n✓ Cell 1 完成")


In [ ]:
# ==========================================
# Cell 2: 下载 Bernini-R 所需模型 (7 个, 约 38GB)
# ------------------------------------------
# 文件名严格对齐工作流 JSON 里 UNETLoader / LoraLoaderModelOnly 的 widget 值。
#
# Bernini-R = Wan2.2 MoE 双专家:
#   high_noise 专家 -> 前期高噪声阶段 (构图/运动/主体替换)
#   low_noise  专家 -> 后期低噪声阶段 (细节/质感)
#
# LoRA 说明:
#   * Bernini-R_LightX2V_high/low_noise -> 社区专为 Bernini-R 训的 4 步蒸馏 LoRA,
#     高/低噪各一个, 强度均 1.0。这是现在推荐的 turbo 方案。
#   * lightx2v_T2V_14B_... -> 原版模板里那个 Wan2.1 T2V LoRA。已弃用,
#     仅为兼容未修改的官方模板而保留下载 (否则前端报缺失模型)。
# ==========================================
import os
import shutil
from huggingface_hub import hf_hub_download
from concurrent.futures import ThreadPoolExecutor

# HF_HUB_ENABLE_HF_TRANSFER 已弃用, 改用 xet 高性能模式
os.environ["HF_XET_HIGH_PERFORMANCE"] = "1"
try:
    from google.colab import userdata
    os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    print("未读到 HF_TOKEN (左侧 Secrets), 公开仓库仍可下载")

M = "/content/ComfyUI/models"
BR = "Comfy-Org/Bernini-R"
KJ = "Kijai/WanVideo_comfy"
WAN = "Comfy-Org/Wan_2.1_ComfyUI_repackaged"
RZ = "rzgar/Bernini-R-LightX2V-4step-loras"

downloads = [
    # --- 主模型 1/2: 高噪专家 (~14.5GB) ---
    {"repo": BR,
     "file": "wan2.2_bernini_r_high_noise_fp8_scaled.safetensors",
     "alt": ["diffusion_models/wan2.2_bernini_r_high_noise_fp8_scaled.safetensors",
             "split_files/diffusion_models/wan2.2_bernini_r_high_noise_fp8_scaled.safetensors"],
     "dir": M + "/diffusion_models"},

    # --- 主模型 2/2: 低噪专家 (~14.5GB) ---
    {"repo": BR,
     "file": "wan2.2_bernini_r_low_noise_fp8_scaled.safetensors",
     "alt": ["diffusion_models/wan2.2_bernini_r_low_noise_fp8_scaled.safetensors",
             "split_files/diffusion_models/wan2.2_bernini_r_low_noise_fp8_scaled.safetensors"],
     "dir": M + "/diffusion_models"},

    # --- 文本编码器 (~6.7GB) ---
    {"repo": WAN,
     "file": "split_files/text_encoders/umt5_xxl_fp8_e4m3fn_scaled.safetensors",
     "dir": M + "/text_encoders"},

    # --- VAE (~250MB) ---
    {"repo": KJ,
     "file": "Wan2_1_VAE_bf16.safetensors",
     "alt": ["VAE/Wan2_1_VAE_bf16.safetensors"],
     "dir": M + "/vae"},

    # --- ⭐ Bernini-R 专用 4 步蒸馏 LoRA: 高噪 (1.24GB) ---
    {"repo": RZ,
     "file": "Bernini-R_LightX2V_high_noise.safetensors",
     "dir": M + "/loras"},

    # --- ⭐ Bernini-R 专用 4 步蒸馏 LoRA: 低噪 (1.24GB) ---
    {"repo": RZ,
     "file": "Bernini-R_LightX2V_low_noise.safetensors",
     "dir": M + "/loras"},

    # --- 已弃用的原版 LoRA (~601MB), 仅为兼容未修改的官方模板保留 ---
    {"repo": KJ,
     "file": "lightx2v_T2V_14B_cfg_step_distill_v2_lora_rank64_bf16.safetensors",
     "alt": ["Lightx2v/lightx2v_T2V_14B_cfg_step_distill_v2_lora_rank64_bf16.safetensors",
             "LoRAs/Lightx2v/lightx2v_T2V_14B_cfg_step_distill_v2_lora_rank64_bf16.safetensors"],
     "dir": M + "/loras"},

    # ==========================================
    # 以下为可选项, 默认不下载
    # ==========================================
    # fp16 满血版双专家 (共约 58GB): G4 96GB 跑得动, 画质略好。
    # {"repo": BR, "file": "wan2.2_bernini_r_high_noise_fp16.safetensors",
    #  "alt": ["diffusion_models/wan2.2_bernini_r_high_noise_fp16.safetensors"],
    #  "dir": M + "/diffusion_models"},
    # {"repo": BR, "file": "wan2.2_bernini_r_low_noise_fp16.safetensors",
    #  "alt": ["diffusion_models/wan2.2_bernini_r_low_noise_fp16.safetensors"],
    #  "dir": M + "/diffusion_models"},
    #
    # int8 / mxfp8 量化版: 给 12-24GB 小显存卡用的, G4 上没必要
    # 1.3B 轻量版 (~2.6GB): 只适合低显存试跑, 效果差很多
]


def fetch(task):
    os.makedirs(task["dir"], exist_ok=True)
    candidates = [task["file"]] + task.get("alt", [])
    name = os.path.basename(task["file"])
    final = os.path.join(task["dir"], name)
    last = None

    if os.path.exists(final):
        print("跳过 (已存在): " + name)
        return

    for cand in candidates:
        try:
            p = hf_hub_download(repo_id=task["repo"], filename=cand, local_dir=task["dir"])
            if os.path.abspath(p) != os.path.abspath(final):
                shutil.move(p, final)
            print("完成: " + name)
            return
        except Exception as e:
            last = e
    print("失败 " + name)
    print("   尝试过的路径: " + str(candidates))
    print("   最后一次报错: " + str(last))


print("开始下载 " + str(len(downloads)) + " 个模型 (共约 38GB, 耗时较长)...")
with ThreadPoolExecutor(max_workers=4) as ex:
    list(ex.map(fetch, downloads))

# ---------------- 核对 ----------------
required = [
    ("diffusion_models", "wan2.2_bernini_r_high_noise_fp8_scaled.safetensors"),
    ("diffusion_models", "wan2.2_bernini_r_low_noise_fp8_scaled.safetensors"),
    ("text_encoders",    "umt5_xxl_fp8_e4m3fn_scaled.safetensors"),
    ("vae",              "Wan2_1_VAE_bf16.safetensors"),
    ("loras",            "Bernini-R_LightX2V_high_noise.safetensors"),
    ("loras",            "Bernini-R_LightX2V_low_noise.safetensors"),
    ("loras",            "lightx2v_T2V_14B_cfg_step_distill_v2_lora_rank64_bf16.safetensors"),
]

print("\n=== 工作流所需文件核对 ===")
missing = []
for sub, name in required:
    fp = os.path.join(M, sub, name)
    if os.path.exists(fp):
        print("  ✓ " + sub + "/" + name +
              "  (" + str(round(os.path.getsize(fp) / 1e9, 2)) + " GB)")
    else:
        print("  ✗ 缺失: " + sub + "/" + name)
        missing.append(sub + "/" + name)

if missing:
    print("\n还缺 " + str(len(missing)) + " 个文件, 前端会报缺失模型。")
    print("   可重跑本格 (已下好的会自动跳过)。")
else:
    print("\n7 个模型全部就绪, 可以跑 Cell 3")


In [ ]:
# ==========================================
# Cell 3: 下载工作流
# 读取位置: ComfyUI 左侧“工作流”面板 (不是“模板”面板)
# ==========================================
import os
import urllib.request

wf_dir = "/content/ComfyUI/user/default/workflows"
os.makedirs(wf_dir, exist_ok=True)

BASE = ("https://raw.githubusercontent.com/Comfy-Org/workflow_templates/"
        "main/templates/")

targets = [
    # 官方原版模板 (未修改, 作对照组用)
    ("video_bernini_r_image_editing.json", BASE + "video_bernini_r_image_editing.json"),
    ("video_bernini_r_video_editing.json", BASE + "video_bernini_r_video_editing.json"),
    # 社区 4 步双 LoRA 工作流 (纯 v2v, 已知能出效果, 用来验证 LoRA 装对了没)
    ("Bernini-R_LightX2V_Workflow.json",
     "https://huggingface.co/rzgar/Bernini-R-LightX2V-4step-loras/"
     "resolve/main/Workflow/Bernini-R_LightX2V_Workflow.json"),
]

for name, url in targets:
    try:
        urllib.request.urlretrieve(url, os.path.join(wf_dir, name))
        print("已下载: " + name)
    except Exception as e:
        print("失败 " + name + ": " + str(e))

print("\n当前 workflows 目录:")
for f in sorted(os.listdir(wf_dir)):
    print("  " + f)

print("\n启动后点左侧边栏的「工作流」面板 (文件夹图标) 打开")
print("修好参数的 rv2v 工作流 bernini_wf_fixed.json 请手动拖进前端加载")


In [ ]:
# ==========================================
# Cell 4: FRP 内网穿透配置 (tcp, 无 token)
# ==========================================
import os
import subprocess

FRP_HOST    = "usoren.usdream.dpdns.org"
FRP_PORT    = 7000
REMOTE_PORT = 8092
FRP_VER     = "0.56.0"
FRP_DIR     = "/content/frp_" + FRP_VER + "_linux_amd64"
ACCESS_URL  = "http://" + FRP_HOST + ":" + str(REMOTE_PORT)

if not os.path.exists(FRP_DIR + "/frpc"):
    print("下载 frp ...")
    subprocess.run(
        "wget -qO- https://github.com/fatedier/frp/releases/download/v"
        + FRP_VER + "/frp_" + FRP_VER + "_linux_amd64.tar.gz | tar -xz -C /content",
        shell=True)
assert os.path.exists(FRP_DIR + "/frpc"), "frpc 下载失败, 请重跑本格"
subprocess.run("chmod +x " + FRP_DIR + "/frpc", shell=True)

frpc_conf = (
    'serverAddr = "' + FRP_HOST + '"\n'
    'serverPort = ' + str(FRP_PORT) + '\n'
    'loginFailExit = false\n'
    'transport.tcpMux = true\n'
    'transport.poolCount = 5\n'
    'log.to = "/content/frpc.log"\n'
    'log.level = "info"\n\n'
    '[[proxies]]\n'
    'name = "bernini_colab"\n'
    'type = "tcp"\n'
    'localIP = "127.0.0.1"\n'
    'localPort = 8188\n'
    'remotePort = ' + str(REMOTE_PORT) + '\n')

with open(FRP_DIR + "/frpc.toml", "w") as f:
    f.write(frpc_conf)

print("frpc.toml 已写入:")
print("-" * 50)
print(frpc_conf)
print("-" * 50)
print("启动后访问: " + ACCESS_URL)
print("地址必须带端口, 不带端口看到的是 frps 自带的 404 页")


In [ ]:
# ==========================================
# Cell 5: 启动 frpc + ComfyUI (重启界面只跑这一格)
# ==========================================
import os
import time
import threading
import subprocess
import configparser

COMFY      = "/content/ComfyUI"
FRP_DIR    = "/content/frp_0.56.0_linux_amd64"
ACCESS_URL = "http://usoren.usdream.dpdns.org:8092"

# sageattention 可选加速。sm_120 上它可能需要现场编译, 失败率不低,
# 所以默认关。cu130 装好后想再榜一点速度再打开。
USE_SAGE = False

assert os.path.isdir(COMFY), "找不到 ComfyUI, 请先跑 Cell 1"
assert os.path.exists(FRP_DIR + "/frpc.toml"), "找不到 frpc.toml, 请先跑 Cell 4"
os.environ["OPENCV_IO_ENABLE_OPENEXR"] = "1"

if USE_SAGE:
    subprocess.run("pip install -q sageattention", shell=True)


def keep_alive():
    while True:
        time.sleep(300)
        print("\n[Keep-Alive] 保持连接活跃...")


threading.Thread(target=keep_alive, daemon=True).start()

for log in ["/content/comfy.log", "/content/frpc.log"]:
    if os.path.exists(log):
        os.remove(log)

# --- 1. 先起 frpc (秒级) ---
print("启动 FRP 穿透...")
subprocess.run("pkill -f '" + FRP_DIR + "/frpc' || true", shell=True)
subprocess.Popen(FRP_DIR + "/frpc -c " + FRP_DIR +
                 "/frpc.toml >> /content/frpc.log 2>&1", shell=True)
time.sleep(6)

frp_log = ""
if os.path.exists("/content/frpc.log"):
    frp_log = open("/content/frpc.log", errors="ignore").read()

if "start proxy success" in frp_log:
    print("FRP 隧道已建立 -> " + ACCESS_URL)
elif "token in login" in frp_log:
    print("服务端开了 token 验证, 请删掉 frps 的 auth.token 后 systemctl restart frps")
elif "already used" in frp_log:
    print("远程端口被占用, 请改 Cell 4 的 REMOTE_PORT")
elif "port not allowed" in frp_log:
    print("端口不在 frps 的 allowPorts 范围内")
else:
    print("frpc 日志:")
print(frp_log[-1200:] if frp_log else "(日志为空)")

# --- 2. 关掉 Manager 启动时的联网 Fetch ---
for cfg in [COMFY + "/user/__manager/config.ini",
            COMFY + "/user/default/ComfyUI-Manager/config.ini",
            COMFY + "/custom_nodes/ComfyUI-Manager/config.ini"]:
    os.makedirs(os.path.dirname(cfg), exist_ok=True)
    c = configparser.ConfigParser()
    if os.path.exists(cfg):
        c.read(cfg)
    if "default" not in c:
        c["default"] = {}
    c["default"]["network_mode"] = "private"
    with open(cfg, "w") as f:
        c.write(f)

# --- 3. 启动 ComfyUI ---
LAUNCH = ("python main.py --listen 127.0.0.1 --port 8188 "
          "--enable-cors-header '*' --preview-method auto")
if USE_SAGE:
    LAUNCH = LAUNCH + " --use-sage-attention"
print("\n启动 ComfyUI...")
print("  " + LAUNCH)
subprocess.Popen(LAUNCH + " > /content/comfy.log 2>&1", shell=True, cwd=COMFY)

ready = False
for i in range(120):
    time.sleep(2)
    if not os.path.exists("/content/comfy.log"):
        continue
    txt = open("/content/comfy.log", errors="ignore").read()
    if "To see the GUI go to" in txt:
        ready = True
        print("ComfyUI ready")
        break
    if "Traceback" in txt and i > 10:
        print("启动报错:")
        print(txt[-3000:])
        break

if not ready:
    print("--- 日志尾部 ---")
    if os.path.exists("/content/comfy.log"):
        print(open("/content/comfy.log", errors="ignore").read()[-3000:])

# --- 4. 优化内核是否真的启用了 ---
if os.path.exists("/content/comfy.log"):
    txt = open("/content/comfy.log", errors="ignore").read()
    if "You need pytorch with cu130" in txt:
        print("\n!! 日志里仍有 cu130 告警 -> 融合内核未启用, 请回去跑 Cell 0")
    else:
        print("\n✓ 未发现 cu130 告警, 融合内核应已生效")

print("\n============================================================")
print("ComfyUI : " + ACCESS_URL)
print("地址必须带 :8092")
print("============================================================\n")

subprocess.run("tail -f /content/comfy.log", shell=True)
